# pybioclip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imageomics/evolution-workshop-2026/blob/main/docs/tutorials/notebooks/pybioclip.ipynb)

## Learning objectives

By the end of this tutorial, you will be able to:

1. Install `pybioclip` and run it on example images.
2. Classify an organism against the full Tree of Life and interpret the ranked predictions.
3. Score an image against your own set of custom labels.
4. Generate image embeddings (feature vectors) for downstream tasks such as similarity search and clustering.

## Prerequisites

- **Python:** >= 3.10 (Colab satisfies this by default).
- **Packages:** `pybioclip` (installed below). Pulls in `torch`, `torchvision`, `open_clip_torch`.
- **Data:** three example images downloaded in the Setup step, so no data prep is required.
- **Prior knowledge:** basic Python. No machine-learning background needed.

## Background

[BioCLIP](https://imageomics.github.io/bioclip/) is a vision foundation model trained on the [TreeOfLife](https://huggingface.co/datasets/imageomics/TreeOfLife-10M) dataset to understand images of organisms across the tree of life. [`pybioclip`](https://imageomics.github.io/pybioclip/) is a small Python library (and CLI) that wraps BioCLIP so you can classify images and extract embeddings in a few lines, with no model-loading or tensor wrangling required.

Two ideas you'll use below:

- **Classification** turns an image into a ranked list of labels with scores. Against the Tree of Life you get taxonomic predictions from kingdom down to species; with custom labels you get scores for terms *you* supply.
- **Embeddings** turn an image into a fixed-length vector that captures its visual content. Similar organisms land near each other in this space, which is the basis for similarity search, clustering, and downstream classifiers.

## Setup

Install `pybioclip` and download three example images: a brown bear, a domestic cat, and a binturong (also called a bearcat).

Image credits: the bear and cat are from the BioCLIP demo; the binturong photo is by Tassilo Rau (Overloon Zoo, 2004), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Binturong_in_Overloon.jpg).

In [ ]:
%pip install -q pybioclip

In [ ]:
import urllib.request

EXAMPLES = {
    "Ursus-arctos.jpeg": "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Ursus-arctos.jpeg",
    "Felis-catus.jpeg": "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Felis-catus.jpeg",
    "Arctictis-binturong.jpeg": "https://upload.wikimedia.org/wikipedia/commons/a/a7/Binturong_in_Overloon.jpg",
}

# A descriptive User-Agent is required by some hosts (e.g. Wikimedia returns
# 403 for the default urllib agent).
HEADERS = {"User-Agent": "evolution-workshop-2026 pybioclip tutorial (https://github.com/Imageomics/evolution-workshop-2026)"}

for name, url in EXAMPLES.items():
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req) as resp, open(name, "wb") as f:
        f.write(resp.read())
    print("downloaded", name)

In [ ]:
from PIL import Image
from IPython.display import display

for name in EXAMPLES:
    img = Image.open(name)
    img.thumbnail((200, 200))  # shrink for display, preserves aspect ratio
    print(name)
    display(img)

## Try it from the command line

Installing `pybioclip` also gives you a `bioclip` command-line tool, so you can classify images and generate embeddings without writing any Python. This is handy for quick checks and for batch-processing a folder of images.

To open a terminal in Colab, click the Terminal button at the bottom-left of the window. The Setup cells above already installed `pybioclip` and downloaded the example images into the notebook's working directory, so first switch into that directory:

```bash
cd /content
```

Then run the commands below at the terminal prompt.

**Classify against the Tree of Life.** Predict the species for the bear image. Add `--format table` for human-readable output instead of the default CSV:

```bash
bioclip predict --rank species --format table Ursus-arctos.jpeg
```

Use `--k` to see more candidates, or `--rank` to predict at a coarser level:

```bash
bioclip predict --rank genus --k 3 --format table Ursus-arctos.jpeg
```

**Classify with your own labels.** Pass a comma-separated list to `--cls`. Here we ask which of `bear`, `cat`, or `bearcat` best matches the binturong, a good zero-shot test since none of these are taxonomic names:

```bash
bioclip predict --cls bear,cat,bearcat --format table Arctictis-binturong.jpeg
```

**Generate embeddings.** Embed one or more images. By default the results print to the screen; redirect them to a file with `>`:

```bash
bioclip embed Ursus-arctos.jpeg Felis-catus.jpeg > embeddings.json
```

Run `bioclip --help`, or `bioclip predict --help`, to see all available options.

## From the command line to Python

The CLI is the fastest way to get predictions, but the Python API gives you full control: you get the raw scores as Python objects, so you can filter, sort, plot, or feed them into the rest of your analysis. The next three steps cover the same three operations you just ran, this time from Python.

## Step 1: Classify against the Tree of Life

`TreeOfLifeClassifier` ranks an image against the full BioCLIP taxonomy. Ask for predictions at a given `Rank` (e.g. `Rank.SPECIES`); each result is a dict with the taxonomic fields plus a `score`.

In [ ]:
from bioclip import TreeOfLifeClassifier, Rank

classifier = TreeOfLifeClassifier()
predictions = classifier.predict("Ursus-arctos.jpeg", Rank.SPECIES)

for p in predictions:
    print(f"{p['species']:30s} ({p['common_name']:20s})  {p['score']:.4f}")

Try a coarser rank, or raise `k` to see more candidates:

In [ ]:
## Step 2: Classify with your own labels

When you only care about a specific set of categories, `CustomLabelsClassifier` scores an image against the labels you provide. Each result is a dict with `classification` and `score`. Here we score the binturong against `bear`, `cat`, and `bearcat`: a zero-shot test, since the model was never trained on these exact words.

from bioclip import CustomLabelsClassifier

classifier = CustomLabelsClassifier(["bear", "cat", "bearcat"])
for p in classifier.predict("Arctictis-binturong.jpeg"):
    print(f"{p['classification']:8s}  {p['score']:.4f}")

In [ ]:
from bioclip import CustomLabelsClassifier

classifier = CustomLabelsClassifier(["bear", "cat", "fish", "bird"])
for p in classifier.predict("Felis-catus.jpeg"):
    print(f"{p['classification']:8s}  {p['score']:.4f}")

## Step 3: Generate image embeddings

`create_image_features` returns a normalized feature vector per image as a `torch.Tensor`. These embeddings are the foundation for similarity search, clustering, and training lightweight downstream classifiers.

In [ ]:
from bioclip import TreeOfLifeClassifier

classifier = TreeOfLifeClassifier()
features = classifier.create_image_features(["Ursus-arctos.jpeg", "Felis-catus.jpeg"])
print("features shape:", tuple(features.shape))  # (num_images, embedding_dim)

Because the vectors are L2-normalized, the dot product between two embeddings is their cosine similarity, a quick check of how visually alike two organisms are:

In [ ]:
bear, cat = features[0], features[1]
similarity = (bear @ cat).item()
print(f"cosine similarity (bear vs. cat): {similarity:.4f}")

## Your turn

_TBD: hands-on prompts for participants. Ideas:_

- _Upload your own image (`Files` panel or `google.colab.files.upload()`) and classify it._
- _Compare predictions at different ranks, or with custom vs. Tree of Life labels._
- _Embed a small folder of images and rank them by similarity to a query image._

## Summary

You installed `pybioclip` and used BioCLIP to (1) classify an image against the Tree of Life, (2) score it against custom labels, and (3) generate image embeddings and measure similarity, all without managing ML infrastructure.

**Next steps**

- [pybioclip documentation](https://imageomics.github.io/pybioclip/): full Python API and CLI reference.
- [Command-line tutorial](https://imageomics.github.io/pybioclip/command-line-tutorial/): `bioclip predict` and `bioclip embed`.
- Other workshop tutorials at [Designing for Discovery](https://imageomics.github.io/evolution-workshop-2026/).

## Troubleshooting

| Problem | Solution |
|---------|----------|
| First prediction is slow | The model weights download once on first use, then are cached for the session. |
| Out-of-memory or very slow | Switch to a GPU runtime using Runtime, then Change runtime type, then GPU. |
| Example image download fails | Re-run the Setup cell, or upload your own image via the Colab `Files` panel. |